# Time Series Analysis — VAR

Model
## RELIANCE.NS | 2005 | With Stationarity Phase

**Pipeline:**
1. Data Collection (yfinance)
2. **Stationarity Testing (ADF Test)**
3. **Differencing + ACF/PACF Analysis**
4. Normalization (MinMaxScaler on stationary data)
5. Train/Test Split
6. AR Model Grid Search
7. Best Model Evaluation
8. Inverse Transform → Original Price Scale
9. Forecasting

---
## Step 1: Install & Import Libraries

In [1]:
!pip install yfinance statsmodels scikit-learn matplotlib --quiet

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

import yfinance as yf
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error, r2_score
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

print("All libraries imported successfully!")

All libraries imported successfully!


---
## Step 2: Download Stock Data

In [3]:
import yfinance as yf
import pandas as pd

ticker = "RELIANCE.NS"

stk_data = yf.download(
    ticker,
    start="2005-01-01",
    end="2023-12-31",
    progress=False
)

if stk_data.empty:
    print("No data available.")
else:
    stk_data = stk_data[["Open", "High", "Low", "Close"]]
    stk_data.dropna(inplace=True)

    print(stk_data.head())
    print("Shape:", stk_data.shape)

Price             Open        High         Low       Close
Ticker     RELIANCE.NS RELIANCE.NS RELIANCE.NS RELIANCE.NS
Date                                                      
2005-01-03   32.366294   32.702319   32.030269   32.633312
2005-01-04   33.002341   33.002341   31.778255   31.877262
2005-01-05   31.562241   32.009272   30.971199   31.799255
2005-01-06   31.682240   32.192276   31.337213   31.574234
2005-01-07   32.102274   32.864328   31.655243   32.537304
Shape: (4688, 4)


In [4]:
from statsmodels.tsa.stattools import adfuller

for col in stk_data.columns:
    result = adfuller(stk_data[col])
    print(col, "p-value:", result[1])

('Open', 'RELIANCE.NS') p-value: 0.9872004818612283
('High', 'RELIANCE.NS') p-value: 0.988301049094881
('Low', 'RELIANCE.NS') p-value: 0.9887666358535843
('Close', 'RELIANCE.NS') p-value: 0.9901483829623715


In [5]:
data_diff = stk_data.diff().dropna()

In [6]:
from statsmodels.tsa.stattools import adfuller

for col in data_diff.columns:
    result = adfuller(data_diff[col])
    print(col, "p-value:", result[1])

('Open', 'RELIANCE.NS') p-value: 0.0
('High', 'RELIANCE.NS') p-value: 0.0
('Low', 'RELIANCE.NS') p-value: 4.225532431315183e-30
('Close', 'RELIANCE.NS') p-value: 0.0


In [15]:
stk_data1=data_diff

In [16]:
from sklearn.preprocessing import MinMaxScaler
Ms = MinMaxScaler()
data1= Ms.fit_transform(stk_data1)
print("Len:",data1.shape)

Len: (4687, 4)


In [17]:
data1=pd.DataFrame(data1,columns=["Open","High","Low","Close"])

In [18]:
training_size = round(len(data1 ) * 0.80)
print(training_size)
X_train=data1[:training_size]
X_test=data1[training_size:]
print("X_train length:",X_train.shape)
print("X_test length:",X_test.shape)
y_train=data1[:training_size]
y_test=data1[training_size:]
print("y_train length:",y_train.shape)
print("y_test length:",y_test.shape)

3750
X_train length: (3750, 4)
X_test length: (937, 4)
y_train length: (3750, 4)
y_test length: (937, 4)


In [21]:
import warnings
warnings.filterwarnings("ignore")

In [22]:
performance={"Model":[],"RMSE":[],"MaPe":[],"Lag":[],"Test":[]}

In [ ]:
import numpy as np

In [49]:
def cominbation(dataset, listt):

    print("Model Columns:", listt)

    datasetTwo = dataset[listt]

    test_obs = 28
    train = datasetTwo[:-test_obs]
    test  = datasetTwo[-test_obs:]

    from statsmodels.tsa.api import VAR
    
    # 🔹 Select Optimal Lag Using AIC
    model = VAR(train)
    x = model.select_order(maxlags=12)
    order = x.selected_orders["aic"]
    
    print("Selected Lag (AIC):", order)

    # 🔹 Fit Model with Selected Lag
    result = model.fit(order)

    # 🔹 Forecast
    lagged_values = train.values[-order:]
    forecast = result.forecast(y=lagged_values, steps=test_obs)

    # 🔹 Convert to DataFrame with Proper Index
    preds = pd.DataFrame(forecast, 
                         index=test.index, 
                         columns=listt)

    preds.to_csv("varforecasted_{}.csv".format(test_obs))

    # 🔹 Evaluation
    from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
    import numpy as np

    rmse = np.sqrt(mean_squared_error(test, preds))
    mape = mean_absolute_percentage_error(test, preds)

    # 🔹 Store Performance
    performance["Model"].append(listt)
    performance["RMSE"].append(round(rmse,4))
    performance["MaPe"].append(round(mape,6))
    performance["Lag"].append(order)
    performance["Test"].append(test_obs)

    perf = pd.DataFrame(performance)

    return perf, result, preds

In [56]:
# Initialize performance dictionary
performance = {
    "Model": [],
    "RMSE": [],
    "MaPe": [],
    "Lag": [],
    "Test": []
}

columns = ['Close', 'High', 'Open', 'Low']

# Progressive comparison
for i in range(2, len(columns)+1):

    current_cols = columns[:i]
    print("\nRunning Model:", current_cols)

    cominbation(data1, current_cols)

# Final result
final_df = pd.DataFrame(performance)

final_df = final_df.sort_values("RMSE").reset_index(drop=True)

print("\nFinal Comparison:")
print(final_df)


Running Model: ['Close', 'High']
Model Columns: ['Close', 'High']
Selected Lag (AIC): 11

Running Model: ['Close', 'High', 'Open']
Model Columns: ['Close', 'High', 'Open']
Selected Lag (AIC): 12

Running Model: ['Close', 'High', 'Open', 'Low']
Model Columns: ['Close', 'High', 'Open', 'Low']
Selected Lag (AIC): 10

Final Comparison:
                      Model    RMSE      MaPe  Lag  Test
0             [Close, High]  0.0362  0.055757   11    28
1  [Close, High, Open, Low]  0.0372  0.057342   10    28
2       [Close, High, Open]  0.0387  0.057990   12    28
